<a href="https://colab.research.google.com/github/pierrenotfound77/PLMezSeq/blob/main/PLMezSeq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**PLMezSeq: ESM-based Tool for Quickly Processing Protein Sequence with Non-standard Amino Acids for PLMs**

<font color="gray">*For Protein Language Models (PLMs) that do not process non-standard amino acids in sequences, this tool helps replace those with most-likely standard ones.*</font>

Input: a csv file containing two columns of data: protein sequence and label OR a single protein string

Output: a csv file process protein sequence with all non-standard amino acids converted to 20 standard ones.
* For U (selenocystein) and O (pyrrolysine): user may choose to delete those entries, or mask it, then predict for a standard amino acid with ESM2 (feature under construction)
* For unknown residue (X, etc.): it will be predicted by ESM2

##**Step1: Upload Your .csv File (Optional)**

Make sure you have included a <font color="red">**"sequence"**</font> column in the csv file :)

In [ ]:
# @title Click the Run Button to upload.
from google.colab import files
import io
import pandas as pd

try:
  # Upload a file from your local machine
  uploaded = files.upload()

  # Assuming the user has uploaded a single CSV file.
  uploaded_filename = None
  for fn in uploaded.keys():
    print('User uploaded file "{name}" with length {length} bytes'.format(
        name=fn, length=len(uploaded[fn])))
    uploaded_filename = fn # Save the filename
    # Read the CSV file into a DataFrame
    df = pd.read_csv(io.StringIO(uploaded[fn].decode('utf-8')))

  # Display the first few rows of the DataFrame
  display(df.head())

  print("You have successfully uploaded " + uploaded_filename)

except:
  print("Upload canceled")

##**Step2: Process the Sequence**

You can either choose the first module to predict for a single string, or choose the second module to run you sequences in your .csv file in batches.

In [ ]:
# @title Predict Non-standard Residues for a Single Protein Sequence
# @markdown After you input the sequence, click the run button
your_sequence = 'AAAXAAAAPPCC' # @param {type:"string"}

import torch
import pandas as pd
from transformers import AutoTokenizer, EsmForMaskedLM
from typing import Tuple


# table for standard amino acids
standard_amino_acids = set("ACDEFGHIKLMNPQRSTVWY")


# use ESM model for predicting masked non-standard amino acids
model = EsmForMaskedLM.from_pretrained("facebook/esm2_t33_650M_UR50D")
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t33_650M_UR50D")


def PLMezSeq(seq: str) -> str:
  """The main PLMezSeq function"""

  # First, produce a masked sequence
  masked_seq, mask_positions = mask_sequence(seq)

  # print(masked_seq)

  # Use ESM2 to predict the masked sequence!
  predicted_seq = predict_sequence(
      seq,
      masked_seq,
      mask_positions
  )

  return predicted_seq


def predict_sequence(original_seq: str, masked_seq: str, mask_positions: list[int]) -> str:
  """Use ESM2 to predict masked amino acids."""

  if len(masked_seq) == 0:
    raise ValueError("Masked sequence cannot be empty.")

  if len(mask_positions) == 0:
    return original_seq

  # use original sequence for replacement
  seq = list(original_seq)


  inputs = tokenizer(
    masked_seq,
    return_tensors="pt"
  )


  with torch.no_grad():
    outputs = model(**inputs)

  logits = outputs.logits[0]

  input_ids = inputs.input_ids[0]

  esm_mask_positions = []

  for i in range(len(input_ids)):
    if input_ids[i] == tokenizer.mask_token_id:
      esm_mask_positions.append(i)


  for i in range(len(mask_positions)):

    original_position = mask_positions[i]

    esm_position = esm_mask_positions[i]


    probabilities = torch.softmax(
      logits[esm_position],
      dim=0
    )

    best_amino_acid = None
    best_probability = 0


    for aa in standard_amino_acids:

      token_id = tokenizer.convert_tokens_to_ids(aa)

      probability = probabilities[token_id]


      if probability > best_probability:
        best_probability = probability
        best_amino_acid = aa


    seq[original_position] = best_amino_acid


  output_seq = "".join(seq)

  return output_seq


def mask_sequence(input_seq: str) -> Tuple[str, list[int]]:
  """
  Process a single protein sequence with ESM2.
    Args:
      input_seq: a protein sequence that possibly has some non-standard amino acids
    Return:
      masked_sequence: a sequence with masked non-standard amino acids
      mask_positions: a list of positions of masked amino acids
  """

  # If given an empty sequence, raise error
  if len(input_seq) == 0:
    print("Input sequence cannot be empty.")
    return "", []

  # Make a residue list
  seq = list(input_seq)

  mask_positions = []

  # Search for positions where a non-standard amino acid exists
  for i in range(len(seq)):
    if seq[i] not in standard_amino_acids:
      mask_positions.append(i)

  # if no non-standard amino acid, return immediately
  if len(mask_positions) == 0:
    return input_seq, []

  # otherwise use ESM tokenizer to mask those positions
  masked_res_list = []

  for i in range(len(seq)):
    if i in mask_positions:
      masked_res_list.append(tokenizer.mask_token)
    else:
      masked_res_list.append(seq[i])

  masked_seq = "".join(masked_res_list)

  return (masked_seq, mask_positions)



#sample_seq1 = "AACKCDDEGPDIRTAPLTGTVDLGSCNAGWEKCASYYTIIADCCRKKKX"
#sample_seq2 = "AACKCDXXXPDIRTAPLTGTVDLGSCNAGWEKCASYYTIIADCCRKKKX"
PLMezSeq(your_sequence)


Loading weights:   0%|          | 0/539 [00:00<?, ?it/s]

'AAAAAAAAPPCC'